# CT-RATE Label Agreement Analysis

Compares LLM-assigned labels (`data/CT-RATE_reports_{model}.csv`) against the original CT-RATE labels (`data/CT-RATE_reports.csv`).  
Metrics are computed per label and overall, stratified by split (train / validation).

In [ ]:
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, cohen_kappa_score, matthews_corrcoef,
    classification_report,
)

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})

DATA_DIR      = Path('../data')
FIGURES_DIR = Path('../figures')
TABLES_DIR = Path('../tables')

## 1. Configuration — select model to evaluate

In [ ]:
# ── Auto-detect available model CSVs ──────────────────────────────────────────
available = sorted(DATA_DIR.glob('CT-RATE_reports_*gpt-5.4.csv'))
print('Available model CSVs:')
for p in available:
    print(' ', p.name)

# Set MODEL_CSV to the file you want to evaluate
# (change if multiple files are present)
MODEL_CSV = available[0] if available else None
print(f'\nUsing: {MODEL_CSV}')

## 2. Load Data

In [ ]:
LABEL_COLS = [
    'Medical material',
    'Arterial wall calcification',
    'Cardiomegaly',
    'Pericardial effusion',
    'Coronary artery wall calcification',
    'Hiatal hernia',
    'Lymphadenopathy',
    'Emphysema',
    'Atelectasis',
    'Lung nodule',
    'Lung opacity',
    'Pulmonary fibrotic sequela',
    'Pleural effusion',
    'Mosaic attenuation pattern',
    'Peribronchial thickening',
    'Consolidation',
    'Bronchiectasis',
    'Interlobular septal thickening',
]

# Reference (original CT-RATE labels)
ref = pd.read_csv(DATA_DIR / 'CT-RATE_reports.csv')

# LLM predictions
llm = pd.read_csv(MODEL_CSV)

# Remove failed extractions
n_total = len(llm)
if '_extraction_failed' in llm.columns:
    llm = llm[~llm['_extraction_failed'].fillna(True).astype(bool)].copy()
    print(f'Extraction failures removed: {n_total - len(llm):,}')

# Rename LLM label cols to avoid collision
llm_renamed = llm[['VolumeName'] + LABEL_COLS].rename(
    columns={c: c + '__llm' for c in LABEL_COLS}
)

# Include text columns for qualitative inspection
text_cols = [c for c in ['Findings_EN', 'Impressions_EN'] if c in ref.columns]
ref_cols = ['VolumeName', 'split'] + text_cols + LABEL_COLS

merged = ref[ref_cols].merge(llm_renamed, on='VolumeName', how='inner')

print(f'Reference rows : {len(ref):,}')
print(f'LLM rows       : {len(llm):,}')
print(f'Matched rows   : {len(merged):,}')
print('\nSplit distribution after merge:')
print(merged['split'].value_counts().to_string())

## 3. Per-label Agreement Metrics

CT-RATE labels and LLM labels are treated symmetrically — neither is assumed to be ground truth. For each label we compute:
- **CT-RATE prevalence** — proportion positive in CT-RATE labels
- **LLM prevalence** — proportion positive in LLM labels
- **Overall agreement** — proportion of cases where both raters agree
- **Positive agreement** — index of specific agreement for the positive category (Cicchetti & Feinstein 1990)
- **Negative agreement** — index of specific agreement for the negative category
- **Discordance rate** — proportion of cases where the two raters disagree
- **Cohen's κ** — chance-corrected agreement

In [ ]:
def per_label_metrics(df_subset, label_cols=LABEL_COLS):
    if len(df_subset) == 0:
        nan_row = {k: np.nan for k in [
            'N', 'Prev_ctrate_%', 'Prev_llm_%',
            'BothPos', 'BothNeg', 'CTpos_LLMneg', 'CTneg_LLMpos',
            'Overall_agreement', 'Positive_agreement', 'Negative_agreement',
            'Discordance_rate', 'Kappa',
        ]}
        return pd.DataFrame(
            [{**{'Label': col}, **nan_row} for col in label_cols]
        ).set_index('Label')

    rows = []
    for col in label_cols:
        y_ct  = df_subset[col].astype(int).values
        y_llm = df_subset[col + '__llm'].astype(int).values
        n = len(y_ct)

        cm = confusion_matrix(y_ct, y_llm, labels=[0, 1])
        both_neg, ct_neg_llm_pos, ct_pos_llm_neg, both_pos = cm.ravel()

        overall_agr = (both_pos + both_neg) / n
        denom_pos   = 2*both_pos + ct_pos_llm_neg + ct_neg_llm_pos
        denom_neg   = 2*both_neg + ct_pos_llm_neg + ct_neg_llm_pos
        pos_agr     = 2*both_pos / denom_pos if denom_pos > 0 else np.nan
        neg_agr     = 2*both_neg / denom_neg if denom_neg > 0 else np.nan
        disc        = (ct_pos_llm_neg + ct_neg_llm_pos) / n

        try:
            kappa = cohen_kappa_score(y_ct, y_llm)
        except Exception:
            kappa = np.nan

        rows.append({
            'Label':              col,
            'N':                  n,
            'Prev_ctrate_%':      round(y_ct.mean()  * 100, 1),
            'Prev_llm_%':         round(y_llm.mean() * 100, 1),
            'BothPos':            int(both_pos),
            'BothNeg':            int(both_neg),
            'CTpos_LLMneg':       int(ct_pos_llm_neg),
            'CTneg_LLMpos':       int(ct_neg_llm_pos),
            'Overall_agreement':  round(overall_agr, 3),
            'Positive_agreement': round(pos_agr, 3) if not np.isnan(pos_agr) else np.nan,
            'Negative_agreement': round(neg_agr, 3) if not np.isnan(neg_agr) else np.nan,
            'Discordance_rate':   round(disc, 3),
            'Kappa':              round(kappa, 3) if not np.isnan(kappa) else np.nan,
        })
    return pd.DataFrame(rows).set_index('Label')


metrics_overall = per_label_metrics(merged)
metrics_train   = per_label_metrics(merged[merged['split'] == 'train'])
metrics_valid   = per_label_metrics(merged[merged['split'] == 'valid'])

print('Table 1. Per-label agreement metrics (overall)\n')
display_cols = ['N', 'Prev_ctrate_%', 'Prev_llm_%',
                'Overall_agreement', 'Positive_agreement', 'Negative_agreement',
                'Discordance_rate', 'Kappa']
print(metrics_overall[display_cols].to_string())

In [ ]:
macro_cols = ['Overall_agreement', 'Positive_agreement', 'Negative_agreement',
              'Discordance_rate', 'Kappa']

summary_rows = []
for subset_name, met in [('Overall', metrics_overall),
                          ('Train',   metrics_train),
                          ('Valid',   metrics_valid)]:
    row = {'Subset': subset_name}
    for c in macro_cols:
        row[c] = round(met[c].mean(), 3)
    summary_rows.append(row)

macro_summary = pd.DataFrame(summary_rows).set_index('Subset')
print('Table 2. Macro-averaged metrics across all 18 labels\n')
print(macro_summary.to_string())

## 4. Figures

### Agreement metrics heatmap

In [ ]:
heat_cols = ['Overall_agreement', 'Positive_agreement', 'Negative_agreement',
             'Discordance_rate', 'Kappa']
heat_data = metrics_overall[heat_cols].copy()
heat_data = heat_data.sort_values('Kappa', ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    heat_data.astype(float),
    annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=0, vmax=1, linewidths=0.4, linecolor='white',
    annot_kws={'size': 8}, cbar_kws={'label': 'Score', 'shrink': 0.7},
    ax=ax,
)
ax.set_title('CT-RATE vs LLM label agreement (sorted by κ)', pad=10)
ax.tick_params(axis='x', rotation=30, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=8)
ax.set_ylabel('')

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'agr_fig1_metrics_heatmap.png', bbox_inches='tight', dpi=200)
plt.show()

### Positive vs negative agreement scatter

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    metrics_overall['Negative_agreement'],
    metrics_overall['Positive_agreement'],
    s=200,
    c=metrics_overall['Kappa'].fillna(0),
    cmap='RdYlGn', vmin=0, vmax=1,
    edgecolors='grey', linewidths=0.5, alpha=0.85,
)

for label, row in metrics_overall.iterrows():
    ax.annotate(
        label, (row['Negative_agreement'], row['Positive_agreement']),
        fontsize=7, ha='left', va='bottom',
        xytext=(4, 3), textcoords='offset points',
    )

plt.colorbar(sc, ax=ax, label="Cohen's κ")
ax.axhline(0.8, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axvline(0.8, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_xlabel('Negative agreement')
ax.set_ylabel('Positive agreement')
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.1)
ax.set_title('Positive vs negative agreement (colour = κ)')

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'agr_fig2_pos_neg_agreement_scatter.png', bbox_inches='tight', dpi=200)
plt.show()

### Agreement breakdown per label (stacked bar)

In [ ]:
breakdown = metrics_overall[['BothPos', 'BothNeg', 'CTpos_LLMneg', 'CTneg_LLMpos']].copy()
breakdown = breakdown.sort_values('BothPos', ascending=False)

n_rows = breakdown.sum(axis=1)
breakdown_pct = breakdown.div(n_rows, axis=0) * 100

colours = {
    'BothPos':      '#4CAF50',
    'BothNeg':      '#2196F3',
    'CTpos_LLMneg': '#FF9800',
    'CTneg_LLMpos': '#F44336',
}
labels_map = {
    'BothPos':      'CT+ / LLM+',
    'BothNeg':      'CT− / LLM−',
    'CTpos_LLMneg': 'CT+ / LLM−',
    'CTneg_LLMpos': 'CT− / LLM+',
}

fig, ax = plt.subplots(figsize=(12, 5))
bottoms = np.zeros(len(breakdown_pct))
x = np.arange(len(breakdown_pct))

for key in ['BothPos', 'BothNeg', 'CTpos_LLMneg', 'CTneg_LLMpos']:
    vals = breakdown_pct[key].values
    ax.bar(x, vals, bottom=bottoms, color=colours[key], label=labels_map[key], width=0.7)
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels(breakdown_pct.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Percentage (%)')
ax.set_ylim(0, 105)
ax.legend(loc='upper right', fontsize=9)
ax.set_title('Agreement breakdown per label')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'agr_fig3_breakdown_stacked.png', bbox_inches='tight', dpi=200)
plt.show()

### Prevalence: CT-RATE vs LLM

In [ ]:
prev_data = metrics_overall[['Prev_ctrate_%', 'Prev_llm_%']].sort_values('Prev_ctrate_%', ascending=False)

x = np.arange(len(prev_data))
w = 0.38

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - w/2, prev_data['Prev_ctrate_%'], width=w,
       label='CT-RATE', color='#4878CF', alpha=0.85)
ax.bar(x + w/2, prev_data['Prev_llm_%'], width=w,
       label='LLM', color='#D65F5F', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(prev_data.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Prevalence (%)')
ax.set_title('Label prevalence: CT-RATE vs LLM')
ax.legend()
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'agr_fig4_prevalence_comparison.png', bbox_inches='tight', dpi=200)
plt.show()

### Cohen's κ: Train vs Validation

In [ ]:
splits_available = merged['split'].unique()
has_train = 'train' in splits_available
has_valid = 'valid' in splits_available

if has_train and has_valid:
    kappa_comp = pd.DataFrame({
        'Train':      metrics_train['Kappa'],
        'Validation': metrics_valid['Kappa'],
    }).sort_values('Train', ascending=False)

    x = np.arange(len(kappa_comp))
    w = 0.38

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(x - w/2, kappa_comp['Train'],      width=w, label='Train',      color='#4878CF', alpha=0.85)
    ax.bar(x + w/2, kappa_comp['Validation'], width=w, label='Validation', color='#D65F5F', alpha=0.85)

    ax.axhline(0.61, color='black', linestyle=':', linewidth=1, alpha=0.6, label='κ = 0.61 (substantial)')
    ax.axhline(0.41, color='grey',  linestyle=':', linewidth=1, alpha=0.6, label='κ = 0.41 (moderate)')
    ax.axhline(0,    color='black', linestyle='-', linewidth=0.6)

    ax.set_xticks(x)
    ax.set_xticklabels(kappa_comp.index, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel("Cohen's κ")
    ax.set_ylim(-0.2, 1.05)
    ax.set_title("Cohen's κ per label — Train vs Validation")
    ax.legend(fontsize=8)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)

    fig.tight_layout()
    plt.savefig(FIGURES_DIR / 'agr_fig5_kappa_train_vs_valid.png', bbox_inches='tight', dpi=200)
    plt.show()
else:
    print(f"Figure skipped: both train and valid splits required. "
          f"Available splits: {list(splits_available)}")

## 5. Confusion Matrices (all 18 labels)

In [ ]:
ncols = 6
nrows = int(np.ceil(len(LABEL_COLS) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.4, nrows * 2.4))
axes_flat = axes.flatten()

for ax, col in zip(axes_flat, LABEL_COLS):
    y_ct  = merged[col].astype(int)
    y_llm = merged[col + '__llm'].astype(int)
    cm = confusion_matrix(y_ct, y_llm, labels=[0, 1])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['LLM−', 'LLM+'],
        yticklabels=['CT-RATE−', 'CT-RATE+'],
        cbar=False, ax=ax, annot_kws={'size': 9},
        linewidths=0.5, linecolor='white',
    )
    kappa = metrics_overall.loc[col, 'Kappa']
    ax.set_title(f'{col}\nκ={kappa:.2f}', fontsize=7, pad=3)
    ax.tick_params(labelsize=7)

for ax in axes_flat[len(LABEL_COLS):]:
    ax.set_visible(False)

fig.suptitle('Confusion matrices — CT-RATE vs LLM (all 18 labels)', y=1.01, fontsize=11)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'agr_fig6_confusion_matrices.png', bbox_inches='tight', dpi=200)
plt.show()

## 6. Summary Table

In [ ]:
def wilson_ci(k, n, z=1.96):
    """Wilson score interval for a proportion k/n."""
    if n == 0:
        return np.nan, np.nan
    p = k / n
    denom  = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    half   = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return max(0, centre - half), min(1, centre + half)


pub_rows = []
slim_rows = []
for col in LABEL_COLS:
    y_ct  = merged[col].astype(int).values
    y_llm = merged[col + '__llm'].astype(int).values
    n = len(y_ct)

    cm = confusion_matrix(y_ct, y_llm, labels=[0, 1])
    both_neg, ct_neg_llm_pos, ct_pos_llm_neg, both_pos = cm.ravel()

    overall_agr = (both_pos + both_neg) / n
    denom_pos   = 2*both_pos + ct_pos_llm_neg + ct_neg_llm_pos
    denom_neg   = 2*both_neg + ct_pos_llm_neg + ct_neg_llm_pos
    pos_agr     = 2*both_pos / denom_pos if denom_pos > 0 else np.nan
    neg_agr     = 2*both_neg / denom_neg if denom_neg > 0 else np.nan
    disc        = (ct_pos_llm_neg + ct_neg_llm_pos) / n

    kappa = cohen_kappa_score(y_ct, y_llm) if len(np.unique(y_ct)) > 1 else np.nan

    agr_lo, agr_hi = wilson_ci(both_pos + both_neg, n)

    pub_rows.append({
        'Label':                  col,
        'N':                      n,
        'CT-RATE prev. (%)':      f'{y_ct.mean()*100:.1f}',
        'LLM prev. (%)':          f'{y_llm.mean()*100:.1f}',
        'CT+/LLM+':               both_pos,
        'CT+/LLM−':               ct_pos_llm_neg,
        'CT−/LLM+':               ct_neg_llm_pos,
        'CT−/LLM−':               both_neg,
        'Overall agreement (95% CI)': (
            f'{overall_agr:.2f} ({agr_lo:.2f}–{agr_hi:.2f})'
        ),
        'Positive agreement':     f'{pos_agr:.2f}' if not np.isnan(pos_agr) else 'N/A',
        'Negative agreement':     f'{neg_agr:.2f}' if not np.isnan(neg_agr) else 'N/A',
        'Discordance rate':       f'{disc:.3f}',
        "Cohen's κ":              f'{kappa:.2f}' if not np.isnan(kappa) else 'N/A',
    })

    slim_rows.append({
        'Label':                      col,
        'Overall agreement (95% CI)': f'{overall_agr*100:.1f}% ({agr_lo*100:.1f}–{agr_hi*100:.1f})',
        'Positive agreement':         f'{pos_agr*100:.1f}%' if not np.isnan(pos_agr) else 'N/A',
        'Negative agreement':         f'{neg_agr*100:.1f}%' if not np.isnan(neg_agr) else 'N/A',
        'Discordance rate':           f'{disc*100:.1f}%',
        "Cohen's κ":                  f'{kappa:.3f}' if not np.isnan(kappa) else 'N/A',
    })

pub_table  = pd.DataFrame(pub_rows)
slim_table = pd.DataFrame(slim_rows)

print('Table 3. Summary agreement table (full)\n')
print(pub_table.to_string(index=False))
print()
print('Table 3b. Summary agreement table (manuscript format)\n')
print(slim_table.to_string(index=False))

In [ ]:
pub_table.to_csv(TABLES_DIR / 'agreement_table.csv', index=False)
slim_table.to_csv(TABLES_DIR / 'agreement_summary_formatted.csv', index=False)
print('Saved: agreement_table.csv')
print('Saved: agreement_summary_formatted.csv')

## 7. Discordant Case Inspection

Sample reports where CT-RATE and LLM labels disagree, useful for qualitative error analysis.

In [ ]:
# ── Set label_to_inspect to any of the 18 label names ────────────────────────
label_to_inspect = 'Atelectasis'
n_samples = 5

ref_col = label_to_inspect
llm_col = label_to_inspect + '__llm'

# CT+ / LLM− : CT-RATE positive, LLM negative
ct_pos_llm_neg_mask = (merged[ref_col] == 1) & (merged[llm_col] == 0)
ct_pos_llm_neg_cases = merged[ct_pos_llm_neg_mask][
    ['VolumeName', 'split', 'Findings_EN', 'Impressions_EN']
].head(n_samples)

# CT− / LLM+ : CT-RATE negative, LLM positive
ct_neg_llm_pos_mask = (merged[ref_col] == 0) & (merged[llm_col] == 1)
ct_neg_llm_pos_cases = merged[ct_neg_llm_pos_mask][
    ['VolumeName', 'split', 'Findings_EN', 'Impressions_EN']
].head(n_samples)

print(f'Label: {label_to_inspect}')
print(f'CT+ / LLM−: {ct_pos_llm_neg_mask.sum():,}')
print(f'CT− / LLM+: {ct_neg_llm_pos_mask.sum():,}')
print()

pd.set_option('display.max_colwidth', 200)
if not ct_pos_llm_neg_cases.empty:
    print('── CT+ / LLM− samples ──')
    display(ct_pos_llm_neg_cases[['VolumeName', 'Findings_EN']].head(3))

if not ct_neg_llm_pos_cases.empty:
    print('── CT− / LLM+ samples ──')
    display(ct_neg_llm_pos_cases[['VolumeName', 'Findings_EN']].head(3))